In [14]:
# =============================================================
# Synthetic Data Generator (Week 1)
# =============================================================
# This notebook generates synthetic data to simulate a budget-
# constrained sequential experimental design setting.
#
# The data has two key properties that make it mathematically
# tractable while remaining realistic:
#   1. SPARSE COEFFICIENTS: out of n=100 features, only k=10
#      have a non-zero effect on the outcome Y. This reflects
#      real-world settings like genomics, where only a few genes
#      out of thousands are relevant for a given outcome.
#
#   2. LOW-RANK COVARIANCE: the features X are not independent
#      — they vary along a shared 10-dimensional latent subspace,
#      captured by a low-rank covariance matrix. This reflects
#      the fact that most real-world high-dimensional data has
#      hidden structure.
#
# Steps:
#   1. Generate sparse coefficient vector alpha (100-dim, 10 non-zero)
#   2. Generate low-rank covariance matrix sigma (100x100, rank 10)
#   3. Sample X from a multivariate Gaussian using sigma (1000 x 100)
#   4. Generate Y = X @ alpha (1000,)
#   5. Verify sample covariance approximates sigma
# =============================================================

In [15]:
import numpy as np
import matplotlib.pyplot as plt 

# set a random seed so results are reproducible 
np.random.seed(42)

# parameters 
n = 100 # number of features
k = 10 # number of non-zero coefficients 
N = 1000 # number of data points

In [16]:
# --- Step 1: Sparse Coefficient Vector ---

# initialize all coefficients to 0 
alpha = np.zeros(n)

# randomly pick 10 indices to be non-zero 
indices = np.random.choice(n, k, replace = False)

#fill those indices with uniform(0,1) values 
alpha[indices] = np.random.uniform(0,1,k)

# verify 
print("non-zero indices:", indices)
print("number of non-zero coefficients:", np.sum(alpha!= 0))
print("alpha vector:", alpha)


non-zero indices: [83 53 70 45 44 39 22 80 10  0]
number of non-zero coefficients: 10
alpha vector: [0.09541012 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.44945067 0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.91495968 0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.65107703 0.         0.
 0.         0.         0.92630088 0.60595997 0.         0.
 0.         0.         0.         0.         0.         0.71134195
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.79017554 0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.85003858 0.         0.         0.1988424
 0.         0.         0.         0.         0.         0.


In [17]:
# --- Step 2: Low-Rank Covariance Matrix ---

# sample a random 10x100 matrix 
M = np.random.randn(k,n)

# outer product gives a 100x100 symmetric low-rank matrix 
sigma = M.T @ M

# verify shape and rank 
print("covariance matrix shape:", sigma.shape)
print("rank of covariance matrix:", np.linalg.matrix_rank(sigma))

covariance matrix shape: (100, 100)
rank of covariance matrix: 10


In [18]:
# --- Step 3: Sample X ---

# sample 1000 data points from a multivariate gaussian using the previous covariance matrix 
X = np.random.multivariate_normal(mean=np.zeros(n), cov=sigma, size=N)

# verify shape 
print("X shape:", X.shape)  # expected: (1000, 100) 

X shape: (1000, 100)


In [19]:
# --- Step 4: Generate Y ---

# generate Y using the sparce coefficient vector 
Y = X @ alpha 

# verify shape 
print("Y shape:", Y.shape) # expected: (1000,)
print("first 10 Y values:", Y[:10])

Y shape: (1000,)
first 10 Y values: [-1.27444391e+01 -1.19617855e+01  7.25374061e-01  3.85867721e+00
  1.98291364e+00 -8.87362124e-04 -2.02799553e+00 -1.17807529e+00
 -9.79341554e+00  1.57176455e+00]


In [20]:
# --- Step 5: Verify ---

# stack X and Y into a full data matrix (1000 x 101)
data = np.column_stack([X,Y])
print("full data matrix shape:", data.shape) # expected: (1000, 101)

# compute samepl covariance of the full data matrix 
sample_cov = np.cov(data.T)
print("sample covariance shape:", sample_cov.shape) # should be (101, 101)

# extract the X submatrix (top-left 100x100 block)
sample_cov_X = sample_cov[:100, :100]

# compare to our generated sigma
difference = np.linalg.norm(sample_cov_X - sigma)
# the difference should be relatively small compared to the scale of sigma itself... check:
print("difference between sample and generated covariance:", difference)
print("norm of sigma:", np.linalg.norm(sigma))

full data matrix shape: (1000, 101)
sample covariance shape: (101, 101)
difference between sample and generated covariance: 38.31451191028047
norm of sigma: 365.7096119878665
